In [72]:
import importlib
import utils
importlib.reload(utils) 
from utils import *

In [2]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error

In [3]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [68]:
kaggle_cred_file = r'E:\env\KAGGLE_KEY\kaggle.json'
load_kaggle_creds(kaggle_cred_file)

In [69]:
competition = 'playground-series-s5e1'
download_competition_data(competition)

Data files already found


In [7]:
import pandas as pd
import numpy as np

In [9]:
train_file = 'data/train.csv'
train_df = pd.read_csv(train_file)
train_df['date'] = pd.to_datetime(train_df['date'])


In [10]:
train_df.head()

,id,date,country,store,product,num_sold
0,0,2010-01-01,Canada,Discount Stickers,Holographic Goose,NaN
1,1,2010-01-01,Canada,Discount Stickers,Kaggle,973.0
2,2,2010-01-01,Canada,Discount Stickers,Kaggle Tiers,906.0
3,3,2010-01-01,Canada,Discount Stickers,Kerneler,423.0
4,4,2010-01-01,Canada,Discount Stickers,Kerneler Dark Mode,491.0


In [11]:
del train_df['id']

In [12]:
train_df.isna().sum()

date           0
country        0
store          0
product        0
num_sold    8871
dtype: int64

In [13]:
category_columns = ['date','country', 'store', 'product']

for col in category_columns:
    print(f'{col:<15} | {train_df[col].nunique()}')

date            | 2557
country         | 6
store           | 3
product         | 5


In [14]:
train_df[['country', 'store', 'product']].drop_duplicates().shape

(90, 3)

In [15]:
train_df.dtypes

date        datetime64[ns]
country             object
store               object
product             object
num_sold           float64
dtype: object

In [16]:
def create_date_features(df, date_col='date'):
    # Extract useful features
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['weekday'] = df['date'].dt.weekday  # Monday=0, Sunday=6
    df['weekday_name'] = df['date'].dt.day_name()
    df['quarter'] = df['date'].dt.quarter
    df['day_of_year'] = df['date'].dt.dayofyear
    df['week_of_year'] = df['date'].dt.isocalendar().week  # ISO week number
    df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)  # 1 if Sat/Sun, else 0
    df['days_in_month'] = df['date'].dt.days_in_month
    df['is_leap_year'] = df['date'].dt.is_leap_year.astype(int)  # 1 if leap year, else 0
    df['week_of_year'] = df['week_of_year'].astype('int32')
    return df

In [17]:
def remove_high_vif_features(X, threshold=5.0):
    """Removes features with VIF above the given threshold iteratively."""
    while True:
        # Calculate VIF for all features
        vif_data = pd.DataFrame()
        vif_data["Feature"] = X.columns
        vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

        # Get the max VIF value and feature
        max_vif = vif_data["VIF"].max()
        if max_vif < threshold:
            break  # Stop if all VIFs are below threshold
        
        # Drop the feature with highest VIF
        feature_to_drop = vif_data.loc[vif_data["VIF"].idxmax(), "Feature"]
        #print(f"Dropping {feature_to_drop} with VIF: {max_vif}")
        X = X.drop(columns=[feature_to_drop])

    return X

In [18]:
def create_train_df(train_df):
    overall_data = train_df.copy()

    try:
        fillna = round(overall_data['num_sold'].mean())
        overall_data['num_sold'] = overall_data['num_sold'].fillna(fillna)
    except Exception as e:
        overall_data['num_sold'] = overall_data['num_sold'].fillna(0)
        print(e)
        print(train_df.shape)
    
    overall_data = overall_data.groupby('date')['num_sold'].sum().reset_index()
    overall_data = create_date_features(overall_data)
    return overall_data

In [19]:
def encode_cyclic_feature(df):
    # Encode cyclic features
    if 'day' in df.columns:
        df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
        df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)
        del df['day']
    if 'week_of_year' in df.columns: 
        df['week_sin'] = np.sin(2 * np.pi * df['week_of_year'] / 53)
        df['week_cos'] = np.cos(2 * np.pi * df['week_of_year'] / 53)
        del df['week_of_year']
    return df

In [49]:
groupby_cols = ['country', 'store', 'product']

model_dict = {}
for index, data in train_df.groupby(groupby_cols):
    #if index == ('Canada', 'Discount Stickers', 'Holographic Goose'):
    #    break
    data = data.drop(columns  = groupby_cols)
    data = create_train_df(data)
    data = create_date_features(data)
    feature_set = [i for i in data.select_dtypes(include=[np.number]).columns if i not in ['date', 'num_sold']] 
    feature_set = remove_high_vif_features(data[feature_set].copy(), threshold=5.0)
    
    feature_set = encode_cyclic_feature(feature_set)

    X_train = feature_set.iloc[:-60]
    X_test = feature_set.iloc[-60:]
    y_train = data['num_sold'][:-60]
    y_test = data['num_sold'][-60:]
    reg = LinearRegression().fit(X_train, y_train)

    model_dict[index] = reg
    y_pred = reg.predict(X_test)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    print(f'{mape:.4f}')

cannot convert float NaN to integer
(2557, 2)
0.0000
0.0881
0.1773
0.1286
0.0924
0.1201
0.0928
0.1917
0.1281
0.0971
0.0484
0.0975
0.1890
0.1228
0.1003
0.0770
0.0704
0.0915
0.0650
0.0704
0.0727
0.0765
0.0986
0.0807
0.0693
0.0684
0.0765
0.1145
0.0745
0.0736
0.0764
0.0676
0.1169
0.0618
0.0696
0.0752
0.0643
0.1226
0.0949
0.0587
0.0816
0.0696
0.1174
0.0842
0.0578
cannot convert float NaN to integer
(2557, 2)
0.0000
0.1738
0.0988
0.1614
0.1394
0.1835
0.1762
0.0863
0.1488
0.1549
0.1481
0.1695
0.0907
0.1253
0.1451
0.2040
0.1542
0.2946
0.2100
0.1651
0.1990
0.1685
0.2643
0.2303
0.1734
0.1899
0.1489
0.2927
0.2124
0.1745
0.0769
0.0901
0.0659
0.0656
0.0808
0.0698
0.0871
0.0828
0.0681
0.0863
0.0768
0.0934
0.0724
0.0718
0.0748


In [50]:
test_file = r'data/test.csv'
test_df = pd.read_csv(test_file)
test_df['date'] = pd.to_datetime(test_df['date'])

In [51]:
test_df.head()

,id,date,country,store,product
0,230130,2017-01-01,Canada,Discount Stickers,Holographic Goose
1,230131,2017-01-01,Canada,Discount Stickers,Kaggle
2,230132,2017-01-01,Canada,Discount Stickers,Kaggle Tiers
3,230133,2017-01-01,Canada,Discount Stickers,Kerneler
4,230134,2017-01-01,Canada,Discount Stickers,Kerneler Dark Mode


In [52]:
model_dict[ ('Canada', 'Discount Stickers', 'Kaggle Tiers')].feature_names_in_

array(['is_weekend', 'is_leap_year', 'day_sin', 'day_cos', 'week_sin',
       'week_cos'], dtype=object)

In [53]:
data.dtypes

date             datetime64[ns]
num_sold                float64
year                      int32
month                     int32
day                       int32
weekday                   int32
weekday_name             object
quarter                   int32
day_of_year               int32
week_of_year              int32
is_weekend                int64
days_in_month             int32
is_leap_year              int64
dtype: object

In [54]:
model_dict

{('Canada', 'Discount Stickers', 'Holographic Goose'): LinearRegression(),
 ('Canada', 'Discount Stickers', 'Kaggle'): LinearRegression(),
 ('Canada', 'Discount Stickers', 'Kaggle Tiers'): LinearRegression(),
 ('Canada', 'Discount Stickers', 'Kerneler'): LinearRegression(),
 ('Canada', 'Discount Stickers', 'Kerneler Dark Mode'): LinearRegression(),
 ('Canada', 'Premium Sticker Mart', 'Holographic Goose'): LinearRegression(),
 ('Canada', 'Premium Sticker Mart', 'Kaggle'): LinearRegression(),
 ('Canada', 'Premium Sticker Mart', 'Kaggle Tiers'): LinearRegression(),
 ('Canada', 'Premium Sticker Mart', 'Kerneler'): LinearRegression(),
 ('Canada', 'Premium Sticker Mart', 'Kerneler Dark Mode'): LinearRegression(),
 ('Canada', 'Stickers for Less', 'Holographic Goose'): LinearRegression(),
 ('Canada', 'Stickers for Less', 'Kaggle'): LinearRegression(),
 ('Canada', 'Stickers for Less', 'Kaggle Tiers'): LinearRegression(),
 ('Canada', 'Stickers for Less', 'Kerneler'): LinearRegression(),
 ('Canad

In [60]:
groupby_cols = ['country', 'store', 'product']
output = []
for index, data in test_df.groupby(groupby_cols):
    result = pd.DataFrame()
    result['id'] = data['id']
    data = data.drop(columns  = groupby_cols)
    #data = create_train_df(data)
    data = create_date_features(data)
    data = encode_cyclic_feature(data)
    model = model_dict[index]
    feature_set = data[model.feature_names_in_]

    y_pred = model.predict(feature_set)
    result['num_sold'] = y_pred

    output.append(result)

In [61]:
final_output = pd.concat(output, axis=0)

In [63]:
final_output = final_output.sort_values(by='id', ascending=True)

In [64]:
final_output.head()

,id,num_sold
0,230130,0.000000
1,230131,829.909331
2,230132,680.058234
3,230133,375.778461
4,230134,422.640787


In [70]:
output_name = 'my_submission_20250225090900.csv'
final_output.to_csv(output_name, index=False)

In [73]:
upload_submission(competition, output_name, 'my first submission')